In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [2]:
spark = (
    SparkSession.builder
        .appName("CryptoETL")
        .config("spark.master", "spark://spark-master:7077")
        # ---- Iceberg + Hive Catalog ----
        .config("spark.sql.catalog.hive_catalog", "org.apache.iceberg.spark.SparkCatalog")
        .config("spark.sql.catalog.hive_catalog.catalog-impl", "org.apache.iceberg.hive.HiveCatalog")
        .config("spark.sql.catalog.hive_catalog.uri", "thrift://hive-metastore:9083")
        .config("spark.sql.catalog.hive_catalog.warehouse", "s3a://crypto-data-lake/")
        # ---- Default catalog
        .config("spark.sql.defaultCatalog", "hive_catalog")
        # ---- S3 (MinIO) ----
        .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
        .config("spark.hadoop.fs.s3a.secret.key", "minioadmin")
        .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        # ---- Iceberg Extensions ----
        .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
        .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
        # ---- Extra JARs ----
        .config("spark.jars", ",".join([
            "/opt/spark-extra-jars/iceberg-spark-runtime-3.5_2.12-1.6.1.jar",
            "/opt/spark-extra-jars/hadoop-aws-3.3.4.jar",
            "/opt/spark-extra-jars/aws-java-sdk-bundle-1.12.262.jar"
        ]))
        .getOrCreate()
)

25/10/05 04:07:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [14]:
bucket = "crypto-data-lake"
landing_date = "2025-10-01"
symbol = "BTCUSDT"
output_path = f"s3a://{bucket}/landing_zone/spot/daily/aggTrades/{symbol}/{landing_date}"
df = spark.read.parquet(output_path)

In [15]:
df = (
    df.withColumn("timestamp_date", F.from_unixtime(F.col("timestamp") / 1_000_000))
    .withColumn("timestamp_second", (F.col("timestamp") / 1_000_000).cast("long"))
    .withColumn("group_id", (F.col("timestamp_second") / 900).cast("long"))
    .withColumn("group_date", F.from_unixtime(F.col("group_id") * 900))
    .withColumn("transform_date", F.current_date())
    .withColumn("transform_timestamp", F.current_timestamp())
    .withColumn("landing_date", F.to_date(F.lit(landing_date), "yyyy-MM-dd"))
    .withColumn("symbol", F.lit(symbol))
)

In [10]:
spark.sql("""
CREATE DATABASE IF NOT EXISTS transform_db
LOCATION 's3a://crypto-data-lake/transform_zone/'
""")

DataFrame[]

In [48]:
from pyspark.errors.exceptions.base import AnalysisException

def table_exists(database: str, table: str) -> bool:
    try:
        spark.catalog.getTable(f"{database}.{table}")
        return True
    except AnalysisException:
        return False

In [51]:
if table_exists("transform_db", "aggtrades"):
    df.writeTo("transform_db.aggtrades").overwritePartitions()
    print("✅ Table exists")
else:
    df.writeTo("transform_db.aggtrades").tableProperty(
    "format-version", "2"
    ).partitionedBy("symbol", "landing_date").createOrReplace()
    print("❌ Table not found")

✅ Table exists


In [23]:
spark.sql("DESCRIBE EXTENDED transform_db.aggtrades").show(truncate=False)

+-----------------------+---------+-------+
|col_name               |data_type|comment|
+-----------------------+---------+-------+
|agg_trade_id           |bigint   |NULL   |
|price                  |double   |NULL   |
|quantity               |double   |NULL   |
|first_trade_id         |bigint   |NULL   |
|last_trade_id          |bigint   |NULL   |
|timestamp              |bigint   |NULL   |
|is_buyer_maker         |boolean  |NULL   |
|is_best_match          |boolean  |NULL   |
|ingest_date            |date     |NULL   |
|ingest_timestamp       |timestamp|NULL   |
|timestamp_date         |string   |NULL   |
|timestamp_second       |bigint   |NULL   |
|group_id               |bigint   |NULL   |
|group_date             |string   |NULL   |
|transform_date         |date     |NULL   |
|transform_timestamp    |timestamp|NULL   |
|landing_date           |date     |NULL   |
|symbol                 |string   |NULL   |
|# Partition Information|         |       |
|# col_name             |data_ty

In [22]:
spark.sql("""
SELECT partition, record_count 
FROM hive_catalog.transform_db.aggtrades.partitions
""").show()

+--------------------+------------+
|           partition|record_count|
+--------------------+------------+
|{BTCUSDT, 2025-10...|     1070020|
+--------------------+------------+



In [25]:
spark.sql("""
CREATE DATABASE IF NOT EXISTS serving_db
LOCATION 's3a://crypto-data-lake/serving_zone/'
""")

DataFrame[]

In [29]:
df_kline = spark.sql("""
select 
    group_id,
    group_date,
    first(timestamp, true) as open_time,
    round(first(price, true), 2) as open_price,
    round(max(price), 2) as high_price,
    round(min(price), 2) as low_price,
    round(last(price, true), 2) as close_price,
    round(sum(quantity), 2) as volume,
    last(timestamp, true) as close_time,
    landing_date,
    symbol
from transform_db.aggtrades
group by group_id, group_date, landing_date, symbol
order by group_id
""")

In [30]:
df_kline.show()

+--------+-------------------+----------------+----------+----------+---------+-----------+------+----------------+------------+-------+
|group_id|         group_date|       open_time|open_price|high_price|low_price|close_price|volume|      close_time|landing_date| symbol|
+--------+-------------------+----------------+----------+----------+---------+-----------+------+----------------+------------+-------+
| 1954752|2025-10-01 00:00:00|1759276800181160| 114048.94|  114308.0|114048.93|  114156.18|117.95|1759277699828046|  2025-10-01|BTCUSDT|
| 1954753|2025-10-01 00:15:00|1759277700016307| 114156.17| 114177.93|113966.67|  114021.86|120.21|1759278598817681|  2025-10-01|BTCUSDT|
| 1954754|2025-10-01 00:30:00|1759278600216478| 114021.85| 114231.26|113969.18|  113969.19| 99.37|1759279499350660|  2025-10-01|BTCUSDT|
| 1954755|2025-10-01 00:45:00|1759279500168353| 113969.19|  114255.0|113969.18|  114239.53| 97.07|1759280398808406|  2025-10-01|BTCUSDT|
| 1954756|2025-10-01 01:00:00|17592804000

In [38]:
if table_exists("serving_db", "klines"):
    df_kline.writeTo("serving_db.klines").overwritePartitions()
    print("✅ Table exists")
else:
    df_kline.writeTo("serving_db.klines").tableProperty(
    "format-version", "2"
    ).partitionedBy("symbol", "landing_date").createOrReplace()
    print("❌ Table not found")

✅ Table exists


In [8]:
spark.sql("""
select group_id, group_date, open_price, high_price, low_price, close_price, volume from serving_db.klines
where landing_date = date('2025-08-02') and symbol = 'BNBUSDT'
order by group_id
""").show()

+--------+-------------------+----------+----------+---------+-----------+-------+
|group_id|         group_date|open_price|high_price|low_price|close_price| volume|
+--------+-------------------+----------+----------+---------+-----------+-------+
| 1948992|2025-08-02 00:00:00|    757.08|     758.3|   755.47|     756.01|2818.61|
| 1948993|2025-08-02 00:15:00|    756.01|    760.94|   756.01|     760.51|2698.99|
| 1948994|2025-08-02 00:30:00|    760.51|    761.84|    759.4|     761.82| 1989.3|
| 1948995|2025-08-02 00:45:00|    761.81|    765.84|    761.8|     765.75|3468.95|
| 1948996|2025-08-02 01:00:00|    765.75|    765.76|   763.04|     763.74|1950.27|
| 1948997|2025-08-02 01:15:00|    763.75|     764.1|   762.37|     762.88| 866.12|
| 1948998|2025-08-02 01:30:00|    762.88|     764.3|    762.5|     763.86| 837.21|
| 1948999|2025-08-02 01:45:00|    763.87|    764.84|   762.98|     764.59| 612.85|
| 1949000|2025-08-02 02:00:00|    764.59|     767.2|   764.37|      767.2|3199.84|
| 19

In [42]:
!jupyter nbconvert --to script end_transform_job.ipynb

[NbConvertApp] Converting notebook end_transform_job.ipynb to script
[NbConvertApp] Writing 4456 bytes to end_transform_job.py


In [7]:
spark.sql("""
select count(*) from transform_db.aggtrades where landing_date = date('2025-10-01')
""").show()

+--------+
|count(1)|
+--------+
| 1070020|
+--------+



In [6]:
spark.sql("""
select count(*) from serving_db.klines
""").show()

+--------+
|count(1)|
+--------+
|     384|
+--------+

